In [2]:
"""
regenerate_heatmaps.py
======================
Regenerates the six dissertation heatmaps with improved colour palettes
so that cell annotations are clearly readable against every background.

Original notebooks are NOT modified. This script reads the same saved
CSV / NPY outputs produced by the frozen pipeline (NB02–NB10) and
writes new PNG files to a dedicated subfolder.

Heatmaps regenerated
--------------------
Main text:
  Figure 4.3  — Station-year natural PM2.5 missingness       (NB02)
  Figure 4.8  — Cross-tier Test MAE by model and scenario     (NB10)

Appendix:
  Figure 8.3  — Fixed top-3 IDW adjacency matrix              (NB09a)
  Figure 8.6  — Candidate-by-scenario Validation MAE          (NB09b)
  Figure 8.7  — Station-level spatial gain from graph messages (NB09c)
  Figure 8.9  — Station-level Test MAE under 72h block gaps   (NB10)

Usage
-----
    python regenerate_heatmaps.py

Output goes to: <CODES_ROOT>/results/regenerated_heatmaps/
"""

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd


# ============================================================
# 0. Locate the project root (same logic as frozen notebooks)
# ============================================================

def find_codes_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path.home() / "Desktop" / "DISSERTATION" / "Codes",
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "results").exists()
            and (candidate / "models").exists()
        ):
            return candidate
        if candidate.name.lower() == "notebooks" and (candidate.parent / "data").exists():
            return candidate.parent
    raise FileNotFoundError(
        "Could not locate the dissertation Codes root. "
        "Run this script from the Codes/ or Codes/notebooks/ directory."
    )


CODES_ROOT = find_codes_root()
RESULTS_ROOT = CODES_ROOT / "results"
PROCESSED_DIR = CODES_ROOT / "data" / "processed"

OUTPUT_DIR = RESULTS_ROOT / "regenerated_heatmaps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Codes root : {CODES_ROOT}")
print(f"Output dir : {OUTPUT_DIR}")


# ============================================================
# 1. Shared constants (mirror NB10 definitions exactly)
# ============================================================

SCENARIOS = [
    "random10_point",
    "random30_point",
    "random50_point",
    "block24_10percent",
    "block72_10percent",
]

SCENARIO_LABELS = {
    "random10_point": "Random 10%",
    "random30_point": "Random 30%",
    "random50_point": "Random 50%",
    "block24_10percent": "Block 24h",
    "block72_10percent": "Block 72h",
}

MODEL_METADATA = {
    "stat_station_mean": {"DisplayName": "Station train mean"},
    "stat_linear_interpolation": {"DisplayName": "Linear interpolation"},
    "stat_kalman_arima": {"DisplayName": "Kalman-ARIMA"},
    "ml_rf_non_spatial": {"DisplayName": "RF non-spatial"},
    "ml_rf_spatio_temporal": {"DisplayName": "RF spatio-temporal"},
    "ml_xgb_non_spatial": {"DisplayName": "XGBoost non-spatial"},
    "ml_xgb_spatio_temporal": {"DisplayName": "XGBoost spatio-temporal"},
    "dl_zero_neighbour": {"DisplayName": "BiGRU zero-neighbour control"},
    "dl_graph_enhanced": {"DisplayName": "Graph-enhanced BiGRU"},
}

MODEL_ORDER = list(MODEL_METADATA)

SHORT_STATION_LABELS = {
    "London Bexley": "Bexley",
    "London Harlington": "Harlington",
    "London Honor Oak Park": "Honor Oak Park",
    "London Marylebone Road": "Marylebone Road",
    "London N. Kensington": "N. Kensington",
}

STATION_ORDER = [
    "London Bexley",
    "London Harlington",
    "London Honor Oak Park",
    "London Marylebone Road",
    "London N. Kensington",
]


# ============================================================
# 2. Adaptive text-colour helper
# ============================================================

def adaptive_text_colour(value: float, vmin: float, vmax: float, cmap) -> str:
    """Return 'white' or 'black' depending on the luminance of the
    background cell colour so that annotations are always readable."""
    if vmax == vmin:
        norm = 0.5
    else:
        norm = (value - vmin) / (vmax - vmin)
    norm = max(0.0, min(1.0, norm))
    rgba = cmap(norm)
    # Perceived luminance (ITU-R BT.601)
    luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
    return "white" if luminance < 0.45 else "black"


# ============================================================
# 3. Figure 4.3 — Station-year natural PM2.5 missingness
# ============================================================

def regenerate_fig_4_3():
    """Reads the EDA panel CSV and reproduces the station-year
    missingness heatmap with a YlOrRd palette."""

    eda_path = PROCESSED_DIR / "selected_pm25_temperature_panel_for_eda.csv"
    if not eda_path.exists():
        eda_path = PROCESSED_DIR / "selected_pm25_temperature_panel.csv"

    eda = pd.read_csv(eda_path)
    eda["RawYear"] = pd.to_datetime(eda["Datetime"]).dt.year

    # Filter to the study period only (matches NB02 expected_years)
    EXPECTED_YEARS = [2021, 2022, 2023, 2024, 2025]
    eda = eda.loc[eda["RawYear"].isin(EXPECTED_YEARS)]

    station_year = (
        eda.groupby(["Station", "RawYear"])
        .agg(
            Total_rows=("PM25", "size"),          # all rows, not just non-null
            PM25_missing=("PM25", lambda x: x.isna().sum()),
        )
        .reset_index()
    )
    station_year["PM25_missing_%"] = round(
        station_year["PM25_missing"] / station_year["Total_rows"] * 100, 1
    )

    matrix = station_year.pivot(
        index="Station", columns="RawYear", values="PM25_missing_%"
    )

    cmap = plt.cm.YlOrRd
    data = matrix.to_numpy()
    vmin, vmax = np.nanmin(data), np.nanmax(data)

    fig, ax = plt.subplots(figsize=(9, 5))
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("PM2.5 missing observations (%)")

    ax.set_xticks(np.arange(len(matrix.columns)), labels=matrix.columns)
    ax.set_yticks(np.arange(len(matrix.index)), labels=matrix.index)
    ax.set_xlabel("Year")
    ax.set_ylabel("Monitoring station")
    ax.set_title("Station-year natural PM2.5 missingness")

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            colour = adaptive_text_colour(val, vmin, vmax, cmap)
            ax.text(
                j, i, f"{val:.1f}",
                ha="center", va="center",
                fontsize=10, fontweight="bold",
                color=colour,
            )

    fig.tight_layout()
    path = OUTPUT_DIR / "fig_4_3_station_year_missingness.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path}")


# ============================================================
# 4. Figure 4.8 — Cross-tier Test MAE heatmap
# ============================================================

def regenerate_fig_4_8():
    """Reads the cross-tier test metrics CSV and reproduces the
    9-model × 5-scenario Test MAE heatmap."""

    ct_dir = RESULTS_ROOT / "final_comparison" / "09_cross_tier"
    metrics_path = ct_dir / "cross_tier_test_metrics.csv"
    metrics_df = pd.read_csv(metrics_path)

    mae_matrix = (
        metrics_df.pivot(index="ModelID", columns="Scenario", values="MAE")
        .reindex(index=MODEL_ORDER, columns=SCENARIOS)
    )

    cmap = plt.cm.YlOrRd
    data = mae_matrix.to_numpy()
    vmin, vmax = np.nanmin(data), np.nanmax(data)

    fig, ax = plt.subplots(figsize=(10.5, 7.2))
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Test MAE (µg/m³)")

    ax.set_xticks(
        np.arange(len(SCENARIOS)),
        labels=[SCENARIO_LABELS[s] for s in SCENARIOS],
        rotation=25, ha="right",
    )
    ax.set_yticks(
        np.arange(len(MODEL_ORDER)),
        labels=[MODEL_METADATA[m]["DisplayName"] for m in MODEL_ORDER],
    )
    ax.set_xlabel("Artificial Test missingness scenario")
    ax.set_ylabel("Frozen imputation configuration")
    ax.set_title("Cross-tier Test MAE by model and missingness scenario")

    for ri in range(data.shape[0]):
        for ci in range(data.shape[1]):
            val = data[ri, ci]
            colour = adaptive_text_colour(val, vmin, vmax, cmap)
            ax.text(
                ci, ri, f"{val:.3f}",
                ha="center", va="center",
                fontsize=8.5, fontweight="bold",
                color=colour,
            )

    fig.tight_layout()
    path = OUTPUT_DIR / "fig_4_8_cross_tier_test_mae_heatmap.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path}")


# ============================================================
# 5. Figure 8.3 — Fixed top-3 IDW adjacency matrix
# ============================================================

def regenerate_fig_8_3():
    """Reads the saved adjacency NPY and reproduces the 5×5
    weight heatmap with a Blues palette."""

    dl_audit_dir = RESULTS_ROOT / "deep_learning" / "08a_data_audit"
    adj_path = dl_audit_dir / "dl_top3_idw_neighbour_adjacency.npy"
    A = np.load(adj_path)

    # Station order from metadata
    meta_path = PROCESSED_DIR / "selected_station_metadata.csv"
    if meta_path.exists():
        meta_df = pd.read_csv(meta_path)
        station_order = meta_df["Station"].astype(str).tolist()
    else:
        station_order = STATION_ORDER

    short_labels = [SHORT_STATION_LABELS.get(s, s) for s in station_order]

    cmap = plt.cm.Blues
    vmin, vmax = 0.0, float(np.max(A))

    fig, ax = plt.subplots(figsize=(8.5, 7))
    im = ax.imshow(A, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Row-normalised IDW weight")

    ax.set_xticks(np.arange(len(short_labels)), labels=short_labels, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(short_labels)), labels=short_labels)
    ax.set_xlabel("Source neighbour station")
    ax.set_ylabel("Target station")
    ax.set_title("Fixed top-3 IDW adjacency matrix\nRows receive messages from columns")

    for ti in range(A.shape[0]):
        for ni in range(A.shape[1]):
            val = A[ti, ni]
            colour = adaptive_text_colour(val, vmin, vmax, cmap)
            ax.text(
                ni, ti, f"{val:.3f}",
                ha="center", va="center",
                fontsize=9, fontweight="bold",
                color=colour,
            )

    fig.tight_layout()
    path = OUTPUT_DIR / "fig_8_3_adjacency_heatmap.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path}")


# ============================================================
# 6. Figure 8.6 — Candidate-by-scenario Validation MAE
# ============================================================

def regenerate_fig_8_6():
    """Reads the ranked candidate summary CSV and reproduces the
    candidate × scenario Validation MAE heatmap."""

    tuning_dir = RESULTS_ROOT / "deep_learning" / "08b_tuning"

    # Find the profile output directory (contains pipeline version subfolder)
    summary_path = None
    for p in sorted(tuning_dir.rglob("dl_ranked_candidate_summary.csv")):
        if "full" in str(p):
            summary_path = p
            break
    if summary_path is None:
        for p in sorted(tuning_dir.rglob("dl_ranked_candidate_summary.csv")):
            summary_path = p
            break

    if summary_path is None:
        print("  SKIPPED Figure 8.6: dl_ranked_candidate_summary.csv not found.")
        return

    summary_df = pd.read_csv(summary_path)
    ordered_df = summary_df.sort_values("CandidateID").reset_index(drop=True)

    scenario_mae_columns = [f"Best_{s}_MAE" for s in SCENARIOS]
    missing_cols = set(scenario_mae_columns) - set(ordered_df.columns)
    if missing_cols:
        # Try alternative column naming
        alt_cols = [f"Best_{s}_MAE" for s in SCENARIOS]
        scenario_mae_columns = [c for c in alt_cols if c in ordered_df.columns]
        if not scenario_mae_columns:
            print(f"  SKIPPED Figure 8.6: missing columns {missing_cols}")
            return

    data = ordered_df[scenario_mae_columns].to_numpy(dtype=float)
    cmap = plt.cm.YlOrRd
    vmin, vmax = np.nanmin(data), np.nanmax(data)

    n_candidates = len(ordered_df)
    fig_height = max(3.5, 0.48 * n_candidates + 1.8)

    fig, ax = plt.subplots(figsize=(9, fig_height))
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Validation MAE (µg/m³)")

    ax.set_xticks(
        np.arange(len(scenario_mae_columns)),
        labels=[SCENARIO_LABELS[s] for s in SCENARIOS[:len(scenario_mae_columns)]],
        rotation=30, ha="right",
    )
    ax.set_yticks(
        np.arange(n_candidates),
        labels=[f"Candidate {cid}" for cid in ordered_df["CandidateID"]],
    )

    for ri in range(data.shape[0]):
        for ci in range(data.shape[1]):
            val = data[ri, ci]
            colour = adaptive_text_colour(val, vmin, vmax, cmap)
            ax.text(
                ci, ri, f"{val:.2f}",
                ha="center", va="center",
                fontsize=8, fontweight="bold",
                color=colour,
            )

    ax.set_title("Candidate-by-scenario Validation MAE (full)")
    fig.tight_layout()
    path = OUTPUT_DIR / "fig_8_6_candidate_scenario_val_mae_heatmap.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path}")


# ============================================================
# 7. Figure 8.7 — Station-level spatial gain (DL)
# ============================================================

def regenerate_fig_8_7():
    """Reads the station-scenario spatial gain CSV and reproduces
    the ΔMAE heatmap with a diverging RdYlGn palette."""

    dl_final_dir = RESULTS_ROOT / "deep_learning" / "08c_final_evaluation"
    gain_path = dl_final_dir / "dl_test_spatial_gain_by_station_scenario.csv"

    if not gain_path.exists():
        print(f"  SKIPPED Figure 8.7: {gain_path.name} not found.")
        return

    gain_df = pd.read_csv(gain_path)

    # Station order from metadata
    meta_path = PROCESSED_DIR / "selected_station_metadata.csv"
    if meta_path.exists():
        meta_df = pd.read_csv(meta_path)
        station_order = meta_df["Station"].astype(str).tolist()
    else:
        station_order = STATION_ORDER

    gain_matrix = (
        gain_df.pivot(
            index="Scenario",
            columns="Station",
            values="DeltaMAE_ZeroMinusGraph",
        )
        .reindex(index=SCENARIOS, columns=station_order)
    )

    data = gain_matrix.to_numpy()
    # Diverging: centre at zero, green = positive (good), red = negative
    abs_max = max(abs(np.nanmin(data)), abs(np.nanmax(data)))
    vmin, vmax = -abs_max, abs_max
    cmap = plt.cm.RdYlGn

    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("ΔMAE: zero-neighbour − graph (µg/m³)")

    short_labels = [s.replace("London ", "") for s in station_order]
    ax.set_xticks(np.arange(len(station_order)), labels=short_labels, rotation=35, ha="right")
    ax.set_yticks(
        np.arange(len(SCENARIOS)),
        labels=[SCENARIO_LABELS[s] for s in SCENARIOS],
    )
    ax.set_xlabel("Station")
    ax.set_ylabel("Missingness scenario")
    ax.set_title("Station-level Test MAE gain from graph messages")

    for ri in range(data.shape[0]):
        for ci in range(data.shape[1]):
            val = data[ri, ci]
            colour = adaptive_text_colour(val, vmin, vmax, cmap)
            ax.text(
                ci, ri, f"{val:.2f}",
                ha="center", va="center",
                fontsize=9, fontweight="bold",
                color=colour,
            )

    fig.tight_layout()
    path = OUTPUT_DIR / "fig_8_7_station_spatial_gain_heatmap.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path}")


# ============================================================
# 8. Figure 8.9 — Station-level Test MAE under 72h block gaps
# ============================================================

def regenerate_fig_8_9():
    """Reads the cross-tier station-level metrics and reproduces
    the 9-model × 5-station block-72h MAE heatmap."""

    ct_dir = RESULTS_ROOT / "final_comparison" / "09_cross_tier"
    station_path = ct_dir / "cross_tier_test_metrics_by_station.csv"

    if not station_path.exists():
        print(f"  SKIPPED Figure 8.9: {station_path.name} not found.")
        return

    station_df = pd.read_csv(station_path)

    block72_matrix = (
        station_df.loc[station_df["Scenario"].eq("block72_10percent")]
        .pivot(index="ModelID", columns="Station", values="MAE")
        .reindex(index=MODEL_ORDER)
    )

    cmap = plt.cm.YlOrRd
    data = block72_matrix.to_numpy()
    vmin, vmax = np.nanmin(data), np.nanmax(data)

    fig, ax = plt.subplots(figsize=(10.5, 7.2))
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Block 72h Test MAE (µg/m³)")

    ax.set_xticks(
        np.arange(len(block72_matrix.columns)),
        labels=block72_matrix.columns,
        rotation=30, ha="right",
    )
    ax.set_yticks(
        np.arange(len(MODEL_ORDER)),
        labels=[MODEL_METADATA[m]["DisplayName"] for m in MODEL_ORDER],
    )
    ax.set_xlabel("AURN station")
    ax.set_ylabel("Model")
    ax.set_title("Station-level Test MAE under 72-hour block gaps")

    for ri in range(data.shape[0]):
        for ci in range(data.shape[1]):
            val = data[ri, ci]
            colour = adaptive_text_colour(val, vmin, vmax, cmap)
            ax.text(
                ci, ri, f"{val:.2f}",
                ha="center", va="center",
                fontsize=8, fontweight="bold",
                color=colour,
            )

    fig.tight_layout()
    path = OUTPUT_DIR / "fig_8_9_block72_station_mae_heatmap.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path}")


if __name__ == "__main__":
    print("\nRegenerating heatmaps with improved colour palettes\n")

    print("[1/6] Figure 4.3 — Station-year missingness")
    regenerate_fig_4_3()

    print("[2/6] Figure 4.8 — Cross-tier Test MAE")
    regenerate_fig_4_8()

    print("[3/6] Figure 8.3 — IDW adjacency matrix")
    regenerate_fig_8_3()

    print("[4/6] Figure 8.6 — Candidate-by-scenario Validation MAE")
    regenerate_fig_8_6()

    print("[5/6] Figure 8.7 — Station-level spatial gain")
    regenerate_fig_8_7()

    print("[6/6] Figure 8.9 — Station-level block-72h MAE")
    regenerate_fig_8_9()

    print(f"\nAll done. Output directory: {OUTPUT_DIR}")

Codes root : C:\Users\Msi\Desktop\DISSERTATION\Codes
Output dir : C:\Users\Msi\Desktop\DISSERTATION\Codes\results\regenerated_heatmaps

Regenerating heatmaps with improved colour palettes...

[1/6] Figure 4.3 — Station-year missingness
  Saved: C:\Users\Msi\Desktop\DISSERTATION\Codes\results\regenerated_heatmaps\fig_4_3_station_year_missingness.png
[2/6] Figure 4.8 — Cross-tier Test MAE
  Saved: C:\Users\Msi\Desktop\DISSERTATION\Codes\results\regenerated_heatmaps\fig_4_8_cross_tier_test_mae_heatmap.png
[3/6] Figure 8.3 — IDW adjacency matrix
  Saved: C:\Users\Msi\Desktop\DISSERTATION\Codes\results\regenerated_heatmaps\fig_8_3_adjacency_heatmap.png
[4/6] Figure 8.6 — Candidate-by-scenario Validation MAE
  Saved: C:\Users\Msi\Desktop\DISSERTATION\Codes\results\regenerated_heatmaps\fig_8_6_candidate_scenario_val_mae_heatmap.png
[5/6] Figure 8.7 — Station-level spatial gain
  Saved: C:\Users\Msi\Desktop\DISSERTATION\Codes\results\regenerated_heatmaps\fig_8_7_station_spatial_gain_heatmap.pn